<a href="https://colab.research.google.com/github/sechali/buku_ajar_ai/blob/main/notebooks/part-12/AI_Modul_12.8_Praktikum_Single_Shot_Detector_(SSD).ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Modul 12.8: Praktikum Single Shot MultiBox Detector (SSD)
**Tujuan Pembelajaran:**
1. Membangun generator Kotak Acuan Default (*Default Boxes*) multi-skala berstandar SSD300.
2. Mengimplementasikan algoritma *Hard Negative Mining* untuk mengatasi ketidakseimbangan kelas latar belakang.
3. Memvisualisasikan distribusi spasial kotak default pada piramida fitur resolusi berbeda.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import math
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as patches

torch.manual_seed(42)
print(f"PyTorch Versi: {torch.__version__}")

### Bagian 1: Implementasi Generator Default Boxes SSD300

In [ ]:
class SSDDefaultBoxesGenerator:
    def __init__(self, img_size=300):
        self.img_size = img_size
        self.feature_maps = [38, 19, 10, 5, 3, 1]
        self.steps = [8, 16, 32, 64, 100, 300]
        self.min_sizes = [30, 60, 111, 162, 213, 264]
        self.max_sizes = [60, 111, 162, 213, 264, 315]
        self.aspect_ratios = [[2], [2, 3], [2, 3], [2, 3], [2], [2]]
        
    def generate(self):
        boxes = []
        for k, f_k in enumerate(self.feature_maps):
            step = self.steps[k]
            for i in range(f_k):
                for j in range(f_k):
                    cx = (j + 0.5) * step / self.img_size
                    cy = (i + 0.5) * step / self.img_size
                    
                    s_k = self.min_sizes[k] / self.img_size
                    boxes.append([cx, cy, s_k, s_k])
                    
                    s_prime = math.sqrt(s_k * (self.max_sizes[k] / self.img_size))
                    boxes.append([cx, cy, s_prime, s_prime])
                    
                    for ar in self.aspect_ratios[k]:
                        w = s_k * math.sqrt(ar)
                        h = s_k / math.sqrt(ar)
                        boxes.append([cx, cy, w, h])
                        boxes.append([cx, cy, h, w])
                        
        return torch.tensor(boxes, dtype=torch.float32)

gen = SSDDefaultBoxesGenerator(img_size=300)
default_boxes = gen.generate()
print(f"Total Default Boxes SSD300: {len(default_boxes):,} kotak")
assert len(default_boxes) == 8732, f"Jumlah kotak salah: {len(default_boxes)}"
print("[VALIDASI SUKSES] 8.732 Kotak Default SSD300 tergenerasi sempurna!")

### Bagian 2: Implementasi Algoritma Hard Negative Mining

In [ ]:
def hard_negative_mining(conf_loss, pos_mask, neg_pos_ratio=3):
    num_pos = pos_mask.long().sum(dim=1, keepdim=True)
    num_neg = torch.clamp(num_pos * neg_pos_ratio, min=1)
    
    conf_loss_neg = conf_loss.clone()
    conf_loss_neg[pos_mask] = -float('inf')
    
    _, indices = conf_loss_neg.sort(dim=1, descending=True)
    _, rank = indices.sort(dim=1)
    
    neg_mask = rank < num_neg
    return neg_mask

# Simulasi batch berukuran 2 dengan 8.732 kotak
mock_conf_loss = torch.rand(2, 8732)
mock_pos_mask = torch.zeros(2, 8732, dtype=torch.bool)
# Misalkan hanya ada 10 kotak positif per citra
mock_pos_mask[0, :10] = True
mock_pos_mask[1, :15] = True

selected_neg_mask = hard_negative_mining(mock_conf_loss, mock_pos_mask, neg_pos_ratio=3)
print(f"Batch 0: Positif={mock_pos_mask[0].sum().item()}, Negatif Terpilih={selected_neg_mask[0].sum().item()} (Rasio 1:3)")
print(f"Batch 1: Positif={mock_pos_mask[1].sum().item()}, Negatif Terpilih={selected_neg_mask[1].sum().item()} (Rasio 1:3)")
assert selected_neg_mask[0].sum().item() == 30, "Sampling negatif gagal!"
print("[VALIDASI SUKSES] Hard Negative Mining menyeimbangkan rasio 3:1 secara presisi!")

### Bagian 3: Visualisasi Kotak Default pada Peta Fitur Resolusi Berbeda

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.set_xlim(0, 1)
ax.set_ylim(1, 0) # Invert Y

# Kotak dari Lapisan Resolusi Tinggi (Conv4_3, k=0) -> Kecil (Biru)
b_small = default_boxes[0].numpy()
rect_s = patches.Rectangle((b_small[0]-b_small[2]/2, b_small[1]-b_small[3]/2), b_small[2], b_small[3],
                          linewidth=2, edgecolor='blue', facecolor='none')
ax.add_patch(rect_s)
ax.text(b_small[0]-b_small[2]/2, b_small[1]-b_small[3]/2-0.02, "Conv4_3 (Objek Mikro)", color='blue', fontsize=8)

# Kotak dari Lapisan Resolusi Menengah (Conv8_2, k=2) -> Sedang (Hijau)
idx_mid = 5776 + 2166 + 300
b_mid = default_boxes[idx_mid].numpy()
rect_m = patches.Rectangle((b_mid[0]-b_mid[2]/2, b_mid[1]-b_mid[3]/2), b_mid[2], b_mid[3],
                          linewidth=2, edgecolor='green', facecolor='none')
ax.add_patch(rect_m)
ax.text(b_mid[0]-b_mid[2]/2, b_mid[1]-b_mid[3]/2-0.02, "Conv8_2 (Objek Sedang)", color='green', fontsize=8)

# Kotak dari Lapisan Resolusi Rendah (Conv11_2, k=5) -> Besar (Merah)
b_large = default_boxes[-1].numpy()
rect_l = patches.Rectangle((b_large[0]-b_large[2]/2, b_large[1]-b_large[3]/2), b_large[2], b_large[3],
                          linewidth=2, edgecolor='red', facecolor='none')
ax.add_patch(rect_l)
ax.text(b_large[0]-b_large[2]/2, b_large[1]-b_large[3]/2-0.02, "Conv11_2 (Objek Makro)", color='red', fontsize=8)

plt.title('Cakupan Skala Kotak Acuan Default SSD Multi-Resolusi')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('praktikum_ssd_default_boxes_coverage_12_8.png', dpi=200)
plt.close()
print("[OK] Grafik cakupan disimpan sebagai 'praktikum_ssd_default_boxes_coverage_12_8.png'")